In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import gurobipy as gp
from gurobipy import GRB
import os

# Step 2

## Task 2.1)

### ALSO-X

In [ ]:
# Load data from generated file
df = pd.read_csv('stochastic_load_profiles.csv', index_col='Scenario_ID')
in_sample = df.iloc[:100].values    # should be (100, 60)
out_of_sample = df.iloc[100:].values  # and this should then be (200, 60)

n_omega, n_minutes = in_sample.shape   # 100, 60
epsilon = 0.1  
M = 10000.0                             

# Budget violation
q = epsilon * n_omega * n_minutes  # 0.1 * 100 * 60 = 600

# Upward flexibility: how much consumption CAN be reduced each minute
F = in_sample - 220   # shape (100, 60), values in [0, 380] kW

# ================= Setup model =====================
model = gp.Model()

# ------------------- variables -----------------------
c_up = model.addVar(lb=0.0, name='c_up')
y = model.addVars(n_omega, n_minutes, vtype=GRB.BINARY, name='y')

# ------------------- Objective function -----------------------
model.setObjective(c_up, GRB.MAXIMIZE)

# --------------------- Constraints -------------------------
for omega in range(n_omega):
    for m in range(n_minutes):
        model.addConstr(c_up - F[omega, m] <= y[omega, m] * M)


model.addConstr(gp.quicksum(y[omega, m]
                            for omega in range(n_omega)
                            for m in range(n_minutes)) <= q)

# ----------------- Optimize --------------------------------------
model.optimize()

if model.Status == GRB.OPTIMAL:
    c_val = c_up.X
    y_vals = np.array([[y[omega, m].X for m in range(n_minutes)]
                        for omega in range(n_omega)])
else:
    print("Reserve model did not solve to optimality.")

print('\n=========== ALSO-X ===============')
print(f"Optimal FCR-D up reserve bid: {c_val:.2f} kW")


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 6001 rows, 6001 columns and 18000 nonzeros (Max)
Model fingerprint: 0xb705bfa6
Model has 1 linear objective coefficients
Variable types: 1 continuous, 6000 integer (6000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-01, 6e+02]

Found heuristic solution: objective -0.0000000
Presolve time: 0.14s
Presolved: 6001 rows, 6001 columns, 18000 nonzeros
Variable types: 1 continuous, 6000 integer (6000 binary)

Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
    5484    1.2459707e+03   3.527980e+03   0.000000e+00      5s
    6001    1.1703390e+03   0.000000e+00   0.000000e+00      6s

Roo

### CVAR

In [21]:
# Set up model
model_cvar = gp.Model()

# -------------- Variables ---------------------------------
c_up_cvar = model_cvar.addVar(lb=0.0, name='c_up')
zeta = model_cvar.addVars(n_omega, n_minutes, lb=-GRB.INFINITY, name='zeta')
beta = model_cvar.addVar(lb=-GRB.INFINITY, ub=0, name='beta') # Gurobi default lb is 0

# -------------------- Objective function ---------------------
model_cvar.setObjective(c_up_cvar, GRB.MAXIMIZE)

# ---------------------------Constraints ------------------
for omega in range(n_omega):
    for m in range(n_minutes):
        model_cvar.addConstr(c_up_cvar - F[omega, m] <= zeta[omega, m])


#total_pairs = n_omega * n_minutes
model_cvar.addConstr(
    (1 / (n_omega * n_minutes)) * gp.quicksum(zeta[omega, m]
                                     for omega in range(n_omega)
                                     for m in range(n_minutes)) <= (1 - epsilon) * beta
)

for omega in range(n_omega):
    for m in range(n_minutes):
        model_cvar.addConstr(beta <= zeta[omega, m])

# ---------- Run model -------------------------
model_cvar.optimize()
print(f"Model status: {model_cvar.Status}")

if model_cvar.Status == GRB.OPTIMAL:
    c_val = c_up_cvar.X
else:
    print("Reserve model did not solve to optimality.")

print('\n=========== CVAR ===============')
print(f"Optimal FCR-D up reserve bid: {c_val:.2f} kW")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 12001 rows, 6002 columns and 30001 nonzeros (Max)
Model fingerprint: 0x82cca290
Model has 1 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-04, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e-01, 4e+02]

Presolve removed 6221 rows and 0 columns
Presolve time: 0.79s
Presolved: 5780 rows, 6002 columns, 23339 nonzeros

Concurrent LP optimizer: dual simplex and barrier
Showing barrier log only...

Ordering time: 0.01s

Barrier statistics:
 Dense cols : 2
 AA' NZ     : 1.734e+04
 Factor NZ  : 2.353e+04 (roughly 5 MB of memory)
 Factor Ops : 1.034e+05 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residua

## Task 2.2)